# Román-style recovery benchmark and reliability-qualified nighttime lights for Samar-Leyte

This notebook tests whether an established nighttime-lights recovery framework can be transferred to Super Typhoon Haiyan and whether reliability qualification improves the resulting functional-recovery signal.

The analysis follows five linked questions:

1. **Benchmark:** What recovery trajectory is produced by a Román-style four-day workflow?
2. **Reliability qualification:** How does the trajectory change when only fresh, high-quality observations over GHSL G7 settlements are retained?
3. **Observability:** Which four-day periods and spatial supports are sufficiently observed?
4. **Functional comparison:** When do the NTL trajectories align with or diverge from NGCP electricity demand?
5. **Communication:** Which simplified figures can communicate the application without concealing uncertainty?

The principal signal is `DNB_BRDF_Corrected_NTL`. Gap-filled NTL is used only as a diagnostic comparison. NTL is interpreted as a proxy for electricity-dependent nocturnal activity, not as a direct measure of electricity restoration or community recovery.

Reference: [Román et al. (2019)](https://doi.org/10.1371/journal.pone.0218883).


In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr

from rasterio.enums import Resampling

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
# ============================================================
# 2. PATHS, EVENT WINDOWS, AND ANALYTICAL SETTINGS
# ============================================================

PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"

A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"

GHSL_CANDIDATES = [
    VNP46_DIR / "GHSL_SMOD_E2015.tif",
    DATA_DIR / "ghsl" / "GHSL_SMOD_E2015.tif",
]

GHSL_PATH = next(
    (path for path in GHSL_CANDIDATES if path.exists()),
    GHSL_CANDIDATES[0],
)

NGCP_CSV_PATH = DATA_DIR / "ngcp" / "NGCP_Hourly_Demand.csv"
REGION_SHAPEFILE = DATA_DIR / "boundaries" / "regions" / "Regions.shp"

# Haiyan and analysis windows
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS = 60
ROMAN_BLOCK_DAYS = 4

ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
ANALYSIS_END = EVENT_DATE + pd.Timedelta(days=365)
PROFILE_END = EVENT_DATE + pd.Timedelta(days=179)

BASELINE_START = EVENT_DATE - pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE - pd.Timedelta(days=1)

STAGE_WINDOWS = {
    "Baseline": (BASELINE_START, PRE_EVENT_END),
    "Stage 1 (0–59 days)": (
        EVENT_DATE,
        EVENT_DATE + pd.Timedelta(days=59),
    ),
    "Stage 2 (60–119 days)": (
        EVENT_DATE + pd.Timedelta(days=60),
        EVENT_DATE + pd.Timedelta(days=119),
    ),
    "Stage 3 (120–179 days)": (
        EVENT_DATE + pd.Timedelta(days=120),
        EVENT_DATE + pd.Timedelta(days=179),
    ),
}

# VNP46A2 bands and spatial dimensions
DNB_BAND = "DNB_BRDF_Corrected_NTL"
GAP_FILLED_BAND = "Gap_Filled_DNB_BRDF_Corrected_NTL"
MQF_BAND = "Mandatory_Quality_Flag"
SPATIAL_DIMS = ("y", "x")

# Reliability and baseline settings
GHSL_G7_CLASSES = (23, 30)
SPATIAL_COMPLETENESS_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0
MIN_BASELINE_COMPOSITES = 3

KNOWN_FILL_VALUES = (
    -9999.0,
    -32768.0,
    6553.5,
    65535.0,
)

# Tacloban display extent
TACLOBAN_X_RANGE = (124.94, 125.10)
TACLOBAN_Y_RANGE = (11.14, 11.32)

# Plot colours retained from the exploratory notebook
SC_COLORSCALE = [
    [0.00, "#F7FCF5"],
    [0.10, "#E5F5E0"],
    [0.25, "#C7E9C0"],
    [0.50, "#74C476"],
    [0.75, "#238B45"],
    [1.00, "#005A32"],
]

DNB_COLOR = "#0091FF"
DNB_FOUR_DAY_COLOR = "#002FFF"
GAP_FILLED_COLOR = "#FF0000"
RQ_COLOR = "#00C54F"
RQ_FOUR_DAY_COLOR = "#009227"
NGCP_COLOR = "#000000"
EVENT_LINE_COLOR = "#0057FF"
STAGE_LINE_COLOR = "#A8B6CC"

for label, path in {
    "VNP46A2": A2_ZARR_PATH,
    "GHSL": GHSL_PATH,
    "NGCP": NGCP_CSV_PATH,
    "Regional boundaries": REGION_SHAPEFILE,
}.items():
    print(f"{label}: {path}")


## 1. Data preparation

The workflow loads daily VIIRS VNP46A2, aligns GHSL settlement classes to the VIIRS grid, and prepares Leyte-Samar NGCP Hour 1 demand. Basic radiance cleaning is applied to both NTL branches. Reliability conditions are introduced later so that the benchmark and reliability-qualified workflows remain distinguishable.


In [ ]:
# ============================================================
# 3.1 LOAD AND CLEAN VNP46A2
# ============================================================

for label, path in {
    "VNP46A2": A2_ZARR_PATH,
    "GHSL": GHSL_PATH,
    "NGCP": NGCP_CSV_PATH,
    "Regional boundaries": REGION_SHAPEFILE,
}.items():
    if not path.exists():
        raise FileNotFoundError(
            f"{label} was not found:\n{path}"
        )


def open_zarr_safely(path):
    try:
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks="auto",
            mask_and_scale=True,
            decode_cf=True,
        )
    except (ImportError, ModuleNotFoundError, ValueError):
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks=None,
            mask_and_scale=True,
            decode_cf=True,
        )


def standardise_date_dimension(ds):
    if "date" not in ds.variables:
        raise KeyError(
            f"No `date` variable found. Variables: {list(ds.variables)}"
        )

    if "date" not in ds.coords:
        ds = ds.set_coords("date")

    observation_dim = ds["date"].dims[0]

    dates = pd.DatetimeIndex(
        pd.to_datetime(ds["date"].values)
    ).normalize()

    ds = ds.assign_coords(
        date=(observation_dim, dates.values)
    )

    if observation_dim != "date":
        ds = ds.swap_dims({observation_dim: "date"})

    return ds.sortby("date")


def prepare_spatial_metadata(ds):
    ds = ds.rio.set_spatial_dims(
        x_dim="x",
        y_dim="y",
        inplace=False,
    )

    if ds.rio.crs is None and "spatial_ref" in ds.variables:
        spatial_attrs = ds["spatial_ref"].attrs

        stored_crs = (
            spatial_attrs.get("crs_wkt")
            or spatial_attrs.get("spatial_ref")
        )

        if stored_crs is not None:
            ds = ds.rio.write_crs(
                stored_crs,
                inplace=False,
            )

    if ds.rio.crs is None:
        x_min = float(ds["x"].min())
        x_max = float(ds["x"].max())
        y_min = float(ds["y"].min())
        y_max = float(ds["y"].max())

        coordinates_are_lonlat = (
            -180 <= x_min <= 180
            and -180 <= x_max <= 180
            and -90 <= y_min <= 90
            and -90 <= y_max <= 90
        )

        if coordinates_are_lonlat:
            ds = ds.rio.write_crs(
                "EPSG:4326",
                inplace=False,
            )
        else:
            raise ValueError(
                "The VNP46A2 CRS could not be recovered."
            )

    return ds


def clean_radiance(values):
    """
    Remove fill values and invalid radiance.

    This is basic data cleaning applied to both methods, not an
    additional reliability filter.
    """
    cleaned = values.astype("float32")

    cleaned = cleaned.where(
        np.isfinite(cleaned)
    )

    fill_values = list(KNOWN_FILL_VALUES)

    for source in (
        values.attrs,
        values.encoding,
    ):
        for key in (
            "_FillValue",
            "missing_value",
        ):
            fill_value = source.get(key)

            if fill_value is not None:
                fill_values.append(fill_value)

    for fill_value in fill_values:
        try:
            fill_value = float(fill_value)

            if np.isfinite(fill_value):
                cleaned = cleaned.where(
                    ~np.isclose(
                        cleaned,
                        fill_value,
                    )
                )
        except (TypeError, ValueError):
            continue

    # Negative DNB radiance is not retained in either profile.
    cleaned = cleaned.where(cleaned >= 0)

    return cleaned


a2 = prepare_spatial_metadata(
    standardise_date_dimension(
        open_zarr_safely(A2_ZARR_PATH)
    )
)

a2 = a2.sel(
    date=slice(
        ANALYSIS_START,
        ANALYSIS_END,
    )
)

missing_bands = [
    band
    for band in (
        DNB_BAND,
        GAP_FILLED_BAND,
        MQF_BAND,
    )
    if band not in a2.data_vars
]

if missing_bands:
    raise KeyError(
        f"Missing A2 bands: {missing_bands}\n"
        f"Available bands: {list(a2.data_vars)}"
    )

dnb = clean_radiance(
    a2[DNB_BAND]
)

mqf = a2[MQF_BAND]

dnb, mqf = xr.align(
    dnb,
    mqf,
    join="inner",
)


print("A2 dimensions:", dict(a2.sizes))
print(
    "Clean DNB range:",
    float(dnb.min(skipna=True).compute()),
    "to",
    float(dnb.max(skipna=True).compute()),
)


In [ ]:
# ============================================================
# 3.2 LOAD AND ALIGN GHSL G7
# ============================================================

ghsl = rxr.open_rasterio(
    GHSL_PATH,
    masked=True,
)

if "band" in ghsl.dims:
    ghsl = ghsl.isel(
        band=0,
        drop=True,
    )

if ghsl.rio.crs is None:
    raise ValueError(
        "The GHSL raster does not contain a CRS."
    )

viirs_template = dnb.isel(
    date=0,
    drop=True,
)

ghsl_viirs = ghsl.rio.reproject_match(
    viirs_template,
    resampling=Resampling.nearest,
)

ghsl_viirs = ghsl_viirs.assign_coords(
    x=viirs_template["x"],
    y=viirs_template["y"],
)

g7_mask = ghsl_viirs.isin(
    GHSL_G7_CLASSES
).fillna(False)

g7_pixel_count = int(
    g7_mask.sum().compute().item()
)

if g7_pixel_count == 0:
    raise ValueError(
        "No GHSL G7 pixels were retained after reprojection."
    )

print(
    "GHSL G7 pixels:",
    f"{g7_pixel_count:,}",
)


In [ ]:
# ============================================================
# 3.3 LOAD NGCP LEYTE-SAMAR HOURLY DEMAND
# ============================================================

if not NGCP_CSV_PATH.exists():
    raise FileNotFoundError(
        f"NGCP CSV not found:\n{NGCP_CSV_PATH}"
    )

# Rows 1–2 contain the dataset title and "Hour No." label.
# Row 3 contains the actual column headings: DATE, 1, 2, ..., 24.
ngcp_raw = pd.read_csv(
    NGCP_CSV_PATH,
    skiprows=2,
    low_memory=False,
)

ngcp_raw.columns = [
    str(column).strip()
    for column in ngcp_raw.columns
]

# Philippine dates in this file are day/month/year.
ngcp_dates = pd.to_datetime(
    ngcp_raw["DATE"].astype(str).str.strip(),
    dayfirst=True,
    errors="coerce",
).dt.normalize()

# Hour 1 = demand during the first hourly interval.
ngcp_hour_1 = pd.to_numeric(
    ngcp_raw["1"],
    errors="coerce",
)

ngcp_daily = (
    pd.DataFrame(
        {
            "date": ngcp_dates,
            "load_mw": ngcp_hour_1,
        }
    )
    .dropna(subset=["date", "load_mw"])
    .groupby("date", as_index=False)["load_mw"]
    .mean()
    .sort_values("date")
)

ngcp_load = (
    ngcp_daily
    .set_index("date")["load_mw"]
    .sort_index()
    .loc[ANALYSIS_START:ANALYSIS_END]
)

if ngcp_load.empty:
    raise ValueError(
        "The NGCP CSV was read, but there are no observations "
        f"between {ANALYSIS_START.date()} and {ANALYSIS_END.date()}."
    )

print(f"NGCP source: {NGCP_CSV_PATH}")
print(f"Available dates: {ngcp_load.index.min().date()} to "
      f"{ngcp_load.index.max().date()}")
print(f"Observations: {len(ngcp_load):,}")
print(f"Missing values: {ngcp_load.isna().sum():,}")

display(ngcp_daily.head())


## 2. Common recovery design

Both NTL branches use the same temporal and recovery calculation:

- a 60-day pre-Haiyan baseline;
- non-overlapping four-day mean composites;
- a per-pixel baseline equal to the median of available pre-event composites;
- identical valid-pixel support in the current and baseline radiance sums; and
- exclusion of composites representing less than 10% of the fixed baseline-lit support.

For composite $i$, recovery is calculated as

$$
Recovery_i = 100\,\frac{\sum_{p \in V_i} NTL_{p,i}}{\sum_{p \in V_i} NTL_{p,0}},
$$

where $V_i$ contains pixels with both a valid current observation and a valid baseline. This common design isolates the effect of the different observation and spatial-support rules.


In [ ]:
# ============================================================
# 4.1 COMMON PIXEL-MATCHED PROFILE FUNCTION
# ============================================================

def build_pixel_matched_profile(
    cube,
    base_mask,
    method,
    input_band,
):
    """
    Calculate four-day NTL recovery using a per-pixel baseline.

    For each composite, the numerator and denominator use exactly
    the same valid pixels:

        Recovery = 100 × Σ(NTL_i) / Σ(NTL_0)
    """

    selected = (
        cube
        .sel(date=slice(ANALYSIS_START, PROFILE_END))
        .where(base_mask)
    )

    dates = pd.DatetimeIndex(
        selected["date"].values
    ).normalize()

    block_numbers = np.floor_divide(
        (dates - EVENT_DATE).days,
        ROMAN_BLOCK_DAYS,
    ).astype(int)

    selected = selected.assign_coords(
        block=("date", block_numbers)
    )

    # Non-overlapping four-day mean composites.
    composites = (
        selected
        .groupby("block")
        .mean(dim="date", skipna=True)
    )

    composite_blocks = (
        composites["block"]
        .values
        .astype(int)
    )

    composite_start = (
        EVENT_DATE
        + pd.to_timedelta(
            composite_blocks * ROMAN_BLOCK_DAYS,
            unit="D",
        )
    )

    composite_end = (
        composite_start
        + pd.Timedelta(days=ROMAN_BLOCK_DAYS - 1)
    )

    baseline_blocks = composite_blocks[
        (composite_start >= BASELINE_START)
        & (composite_end <= PRE_EVENT_END)
    ]

    if len(baseline_blocks) == 0:
        raise ValueError(
            f"{method}: no four-day baseline composites were found."
        )

    # Per-pixel NTL0: median of pre-Haiyan four-day composites.
    baseline_composites = composites.sel(
        block=baseline_blocks
    )

    baseline_observations = (
        baseline_composites
        .notnull()
        .sum(dim="block")
    )

    ntl0 = (
        baseline_composites
        .median(dim="block", skipna=True)
        .compute()
    )

    fixed_mask = (
        base_mask
        & (baseline_observations >= 1)
        & np.isfinite(ntl0)
        & (ntl0 > 0)
    ).compute()

    fixed_pixel_count = int(
        fixed_mask.sum().item()
    )

    if fixed_pixel_count == 0:
        raise ValueError(
            f"{method}: no valid baseline-lit pixels were found."
        )

    # Use matching pixels in NTL_i and NTL_0.
    paired_valid = (
        composites.notnull()
        & fixed_mask
        & ntl0.notnull()
    )

    valid_pixel_count = paired_valid.sum(
        dim=SPATIAL_DIMS
    )

    spatial_coverage_pct = (
        100.0
        * valid_pixel_count
        / fixed_pixel_count
    )

    current_radiance = (
        composites
        .where(paired_valid)
        .sum(
            dim=SPATIAL_DIMS,
            skipna=True,
            min_count=1,
        )
    )

    matched_baseline_radiance = (
        ntl0
        .where(paired_valid)
        .sum(
            dim=SPATIAL_DIMS,
            skipna=True,
            min_count=1,
        )
    )

    recovery_pct = (
        100.0
        * current_radiance
        / matched_baseline_radiance
    )

    reduced = xr.Dataset(
        {
            "current_radiance": current_radiance,
            "matched_baseline_radiance": (
                matched_baseline_radiance
            ),
            "recovery_pct": recovery_pct,
            "spatial_coverage_pct": (
                spatial_coverage_pct
            ),
            "valid_pixel_count": valid_pixel_count,
        }
    ).compute()

    profile = (
        reduced
        .to_dataframe()
        .reset_index()
        .sort_values("block")
        .reset_index(drop=True)
    )

    profile["date_start"] = (
        EVENT_DATE
        + pd.to_timedelta(
            profile["block"] * ROMAN_BLOCK_DAYS,
            unit="D",
        )
    )

    profile["date_end"] = (
        profile["date_start"]
        + pd.Timedelta(days=ROMAN_BLOCK_DAYS - 1)
    )

    # Do not interpret composites below 10% spatial support.
    profile.loc[
        profile["spatial_coverage_pct"]
        < SPATIAL_COMPLETENESS_PCT,
        [
            "current_radiance",
            "matched_baseline_radiance",
            "recovery_pct",
        ],
    ] = np.nan

    profile["method"] = method

    baseline_profile_mask = (
        (profile["date_start"] >= BASELINE_START)
        & (profile["date_end"] <= PRE_EVENT_END)
    )

    report = {
        "method": method,
        "input_band": input_band,
        "baseline_start": BASELINE_START.date(),
        "baseline_end": PRE_EVENT_END.date(),
        "baseline_composites": len(baseline_blocks),
        "fixed_pixels": fixed_pixel_count,
        "minimum_completeness_pct": (
            SPATIAL_COMPLETENESS_PCT
        ),
        "median_baseline_coverage_pct": (
            profile.loc[
                baseline_profile_mask,
                "spatial_coverage_pct",
            ].median()
        ),
        "median_pixel_ntl0": float(
            ntl0
            .where(fixed_mask)
            .median(
                dim=SPATIAL_DIMS,
                skipna=True,
            )
            .item()
        ),
    }

    return (
        profile,
        composites.where(fixed_mask),
        ntl0.where(fixed_mask),
        fixed_mask,
        report,
    )


## 3. Román-style benchmark

The benchmark transfers the published event-fidelity logic to Samar-Leyte using directly observed DNB-BRDF radiance, basic fill-value screening, baseline-lit spatial support, and four-day aggregation. It does not apply MQF or GHSL filtering. Gap-filled radiance is retained only to show how reconstruction changes the daily record.


In [ ]:
# ============================================================
# 4.2 ROMÁN-STYLE PIXEL-MATCHED PROFILE
# ============================================================

# ------------------------------------------------------------
# Román-style input: directly observed DNB-BRDF
# ------------------------------------------------------------

roman_cube = dnb

# No GHSL or MQF restriction in the Román-style branch.
roman_base_mask = xr.ones_like(
    roman_cube.isel(date=0),
    dtype=bool,
)

(
    roman_profile,
    roman_composites,
    roman_ntl0,
    roman_fixed_mask,
    roman_report,
) = build_pixel_matched_profile(
    cube=roman_cube,
    base_mask=roman_base_mask,
    method="Román-style transfer",
    input_band=(
        "Direct DNB_BRDF_Corrected_NTL; "
        "basic fill-value screening"
    ),
)

print("Román-style pixel-matched profile")
print(f"Input: {roman_report['input_band']}")
print(
    f"Baseline: {BASELINE_START.date()} to "
    f"{PRE_EVENT_END.date()}"
)
print(
    "Baseline composites: "
    f"{roman_report['baseline_composites']}"
)
print(
    "Baseline-lit pixels: "
    f"{roman_report['fixed_pixels']:,}"
)
print(
    "Median baseline coverage: "
    f"{roman_report['median_baseline_coverage_pct']:.1f}%"
)

display(roman_profile.head())


In [ ]:
# ============================================================
# 4.3 NGCP FOUR-DAY COMPARATOR
# ============================================================

ngcp_series = (
    ngcp_load
    .loc[ANALYSIS_START:PROFILE_END]
    .dropna()
    .astype(float)
)

ngcp_blocks = np.floor_divide(
    (ngcp_series.index - EVENT_DATE).days,
    ROMAN_BLOCK_DAYS,
).astype(int)

ngcp_composites = (
    ngcp_series
    .groupby(ngcp_blocks)
    .mean()
)

ngcp_profile = pd.DataFrame(
    {
        "block": ngcp_composites.index.astype(int),
        "load_mw": ngcp_composites.values,
    }
)

ngcp_profile["date_start"] = (
    EVENT_DATE
    + pd.to_timedelta(
        ngcp_profile["block"] * ROMAN_BLOCK_DAYS,
        unit="D",
    )
)

ngcp_profile["date_end"] = (
    ngcp_profile["date_start"]
    + pd.Timedelta(days=ROMAN_BLOCK_DAYS - 1)
)

ngcp_baseline_mask = (
    (ngcp_profile["date_start"] >= BASELINE_START)
    & (ngcp_profile["date_end"] <= PRE_EVENT_END)
)

ngcp_baseline_values = (
    ngcp_profile.loc[
        ngcp_baseline_mask,
        "load_mw",
    ]
    .dropna()
)

if len(ngcp_baseline_values) < MIN_BASELINE_COMPOSITES:
    raise ValueError(
        "Insufficient NGCP four-day composites in the "
        "60-day pre-Haiyan baseline."
    )

ngcp_baseline_reference = (
    ngcp_baseline_values.median()
)

ngcp_profile["recovery_pct"] = (
    100.0
    * ngcp_profile["load_mw"]
    / ngcp_baseline_reference
)

print(
    "NGCP baseline reference: "
    f"{ngcp_baseline_reference:.3f} MW "
    f"from {len(ngcp_baseline_values)} composites"
)


### 3.1 Daily observations, gap filling, and four-day aggregation

The next figures move from the daily record to the four-day benchmark. Spatial completeness remains visible because a nearly continuous composite series can still contain periods with weak ground observation.


In [ ]:
# ============================================================
# 5.1 PREPARE DAILY AND FOUR-DAY DIAGNOSTIC SERIES
# ============================================================

gap_filled_cube = clean_radiance(
    a2[GAP_FILLED_BAND]
)


def extract_daily_ntl_and_sc(cube, fixed_mask):
    """Return daily regional mean NTL and spatial completeness."""

    fixed_mask = fixed_mask.fillna(False).astype(bool)

    fixed_pixel_count = int(
        fixed_mask.sum().compute().item()
    )

    if fixed_pixel_count == 0:
        raise ValueError("The fixed mask contains no pixels.")

    selected = (
        cube
        .sel(date=slice(ANALYSIS_START, PROFILE_END))
        .where(fixed_mask)
    )

    valid_pixel_count = selected.notnull().sum(
        dim=SPATIAL_DIMS
    )

    daily_data = xr.Dataset(
        {
            "mean_ntl": selected.mean(
                dim=SPATIAL_DIMS,
                skipna=True,
            ),
            "sc_pct": (
                100.0
                * valid_pixel_count
                / fixed_pixel_count
            ),
            "valid_pixel_count": valid_pixel_count,
        }
    ).compute()

    daily_frame = (
        daily_data
        .to_dataframe()
        .reset_index()
    )

    daily_frame["date"] = pd.to_datetime(
        daily_frame["date"]
    ).dt.normalize()

    return (
        daily_frame
        .groupby("date", as_index=False)
        .mean(numeric_only=True)
        .sort_values("date")
        .set_index("date")
    )


roman_daily = extract_daily_ntl_and_sc(
    cube=roman_cube,
    fixed_mask=roman_fixed_mask,
)

gap_filled_daily = extract_daily_ntl_and_sc(
    cube=gap_filled_cube,
    fixed_mask=roman_fixed_mask,
)

roman_profile["raw_mean_ntl"] = (
    roman_profile["current_radiance"]
    / roman_profile["valid_pixel_count"]
)


In [ ]:
# ============================================================
# 5.2 SHARED DIAGNOSTIC PLOTTING FUNCTION
# ============================================================

def plot_sc_timeseries(
    title,
    ntl_panel_title,
    sc_series,
    line_series,
    y_axis_title,
):
    """
    Plot spatial-completeness strips above a time series.

    sc_series: list of dictionaries containing label, x and y.
    line_series: list of dictionaries containing the line settings.
    """

    figure = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        row_heights=[
            0.10,
            0.85,
        ],
        subplot_titles=(
            "Spatial Completeness (SC)",
            ntl_panel_title,
        ),
    )

    # Spatial-completeness strips
    # Align all SC series to a shared date axis before plotting.

    sc_dates = pd.DatetimeIndex(
        sorted(
            set().union(
                *[
                    pd.to_datetime(
                        sc_item["x"]
                    ).tolist()
                    for sc_item in sc_series
                ]
            )
        )
    )

    sc_labels = []
    sc_matrix = []

    for sc_item in sc_series:
        sc_frame = pd.DataFrame(
            {
                "date": pd.to_datetime(
                    sc_item["x"]
                ),
                "sc_pct": np.asarray(
                    sc_item["y"],
                    dtype=float,
                ),
            }
        )

        sc_frame = (
            sc_frame
            .groupby("date", as_index=True)["sc_pct"]
            .mean()
            .reindex(sc_dates)
        )

        sc_labels.append(
            sc_item["label"]
        )

        sc_matrix.append(
            sc_frame.to_numpy(
                dtype=float
            )
        )

    figure.add_trace(
        go.Heatmap(
            x=sc_dates,
            y=sc_labels,
            z=np.vstack(sc_matrix),
            coloraxis="coloraxis",
            zsmooth=False,
            hoverongaps=False,
            hovertemplate=(
                "%{x|%d %b %Y}<br>"
                "Support: %{y}<br>"
                "Spatial completeness: %{z:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )

    # Time-series lines
    for line_item in line_series:
        figure.add_trace(
            go.Scatter(
                x=pd.to_datetime(line_item["x"]),
                y=line_item["y"],
                mode=line_item.get(
                    "mode",
                    "lines",
                ),
                name=line_item["name"],
                connectgaps=False,
                opacity=line_item.get(
                    "opacity",
                    1.0,
                ),
                line=dict(
                    color=line_item["color"],
                    width=line_item.get(
                        "width",
                        2.0,
                    ),
                    dash=line_item.get(
                        "dash",
                        "solid",
                    ),
                    shape=line_item.get(
                        "shape",
                        "linear",
                    ),
                ),
                marker=dict(
                    size=line_item.get(
                        "marker_size",
                        4,
                    ),
                ),
                hovertemplate=(
                    "%{x|%d %b %Y}<br>"
                    f"{line_item['name']}: "
                    "%{y:.2f} "
                    f"{line_item['unit']}"
                    "<extra></extra>"
                ),
            ),
            row=2,
            col=1,
        )

    # Haiyan and fixed 60-day intervals
    for row_number in (1, 2):
        figure.add_vline(
            x=EVENT_DATE.to_pydatetime(),
            line=dict(
                color=EVENT_LINE_COLOR,
                width=2,
                dash="dash",
            ),
            row=row_number,
            col=1,
        )

        for boundary_day in (60, 120):
            figure.add_vline(
                x=(
                    EVENT_DATE
                    + pd.Timedelta(
                        days=boundary_day
                    )
                ).to_pydatetime(),
                line=dict(
                    color=STAGE_LINE_COLOR,
                    width=1.5,
                    dash="dot",
                ),
                row=row_number,
                col=1,
            )

    figure.add_annotation(
        x=(EVENT_DATE + pd.Timedelta(days=5)).to_pydatetime(),
        y=0.05,
        xref="x2",
        yref="y2 domain",
        text="Haiyan",
        showarrow=False,
        xanchor="left",
        font=dict(
            color=EVENT_LINE_COLOR,
            size=16,
        ),
    )

    sc_labels = [
        item["label"]
        for item in sc_series
    ]

    figure.update_yaxes(
        categoryorder="array",
        categoryarray=sc_labels[::-1],
        row=1,
        col=1,
    )

    figure.update_yaxes(
        title_text=y_axis_title,
        row=2,
        col=1,
    )

    figure.update_xaxes(
        range=[
            ANALYSIS_START,
            PROFILE_END,
        ],
    )

    figure.update_xaxes(
        title_text="Date",
        row=2,
        col=1,
    )

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=1200,
        height=600,
        title=dict(
            text=title,
            x=0.5,
            y=0.92,
            xanchor="center",
            font=dict(size=24),
        ),
        legend=dict(
            orientation="h",
            yanchor="middle",
            y=0.8,
            xanchor="center",
            x=0.8,
            font=dict(size=16),
        ),
        coloraxis=dict(
            colorscale=SC_COLORSCALE,
            cmin=0,
            cmax=100,
            colorbar=dict(
                title=dict(
                    text="SC (%)"
                ),
                x=1.015,
                xanchor="left",
                y=0.98,
                len=0.21,
                thickness=18,
                outlinewidth=0.8,
                outlinecolor="#555555",
            ),
        ),
        font=dict(
            family="Arial",
            size=16,
            color="#243B5A",
        ),
        margin=dict(
            l=105,
            r=105,
            t=140,
            b=70,
        ),
        hovermode="x",
    )

    return figure


In [ ]:
# ------------------------------------------------------------
# ROMÁN 1:
# Raw daily direct DNB-BRDF versus daily gap-filled NTL
# ------------------------------------------------------------

fig_roman_raw = plot_sc_timeseries(
    title=(""
        # "Daily raw and gap-filled DNB-BRDF NTL"
    ),
    ntl_panel_title=(""
        # "Daily Regional Mean NTL Radiance"
    ),
    sc_series=[
        {
            "label": "",
            "x": roman_daily.index,
            "y": roman_daily["sc_pct"],
        },
    ],
    line_series=[
        {
            "name": "DNB-BRDF",
            "x": roman_daily.index,
            "y": roman_daily["mean_ntl"],
            "color": DNB_COLOR,
            "width": 1.5,
            "mode": "lines+markers",
            "marker_size": 3,
            "opacity": 0.80,
            "unit": "nW cm⁻² sr⁻¹",
        },
        {
            "name": "Gap-filled DNB-BRDF",
            "x": gap_filled_daily.index,
            "y": gap_filled_daily["mean_ntl"],
            "color": GAP_FILLED_COLOR,
            "width": 2.0,
            "mode": "lines",
            "unit": "nW cm⁻² sr⁻¹",
        },
    ],
    y_axis_title=(
        "Mean DNB-BRDF<br>"
        "(nW cm⁻² sr⁻¹)"
    ),
)

fig_roman_raw.update_yaxes(type='log',row=2, col=1)
fig_roman_raw.show()


**Interpretation.** Gap filling smooths the daily series but does not create new direct observations. Differences around landfall are therefore diagnostic rather than additional evidence of the event state.


In [ ]:
# ------------------------------------------------------------
# ROMÁN 2:
# Raw daily DNB-BRDF versus four-day aggregation
# ------------------------------------------------------------

fig_roman_aggregation = plot_sc_timeseries(
    title="",#"Daily BRDF to Four-Day Aggregation",
    ntl_panel_title=(""
        # "Raw daily DNB-BRDF versus four-day mean"
    ),
    sc_series=[
        {
            "label": "",
            "x": roman_profile["date_start"],
            "y": roman_profile["spatial_coverage_pct"],
        },
    ],
    line_series=[
        {
            "name": "Raw daily DNB-BRDF",
            "x": roman_daily.index,
            "y": roman_daily["mean_ntl"],
            "color": DNB_COLOR,
            "width": 1.3,
            "mode": "lines+markers",
            "marker_size": 3,
            "opacity": 0.60,
            "unit": "nW cm⁻² sr⁻¹",
        },
        {
            "name": "Four-day mean DNB-BRDF",
            "x": roman_profile["date_start"],
            "y": roman_profile["raw_mean_ntl"],
            "color": DNB_FOUR_DAY_COLOR,
            "width": 2.8,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "nW cm⁻² sr⁻¹",
        },
    ],
    y_axis_title=(
        "Mean DNB-BRDF<br>"
        "(nW cm⁻² sr⁻¹)"
    ),
)

fig_roman_aggregation.update_yaxes(type='log',row=2, col=1)
fig_roman_aggregation.show()


**Interpretation.** Four-day aggregation improves temporal continuity, but the completeness strip shows that observation support remains uneven. Temporal retention and spatial support must therefore be assessed separately.


In [ ]:
# ------------------------------------------------------------
# ROMÁN 3:
# Baseline-normalized four-day DNB versus four-day NGCP
# ------------------------------------------------------------

fig_roman_normalized = plot_sc_timeseries(
    title=(""
        # "Baseline-normalized "
        # "NTL and NGCP"
    ),
    ntl_panel_title=(""
        # "Four-day output relative to pre-Haiyan baseline"
    ),
    sc_series=[
        {
            "label": "",
            "x": roman_profile["date_start"],
            "y": roman_profile[
                "spatial_coverage_pct"
            ],
        },
    ],
    line_series=[
        {
            "name": "DNB-BRDF (4D)",
            "x": roman_profile["date_start"],
            "y": roman_profile["recovery_pct"],
            "color": DNB_FOUR_DAY_COLOR,
            "width": 2.8,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "%",
        },
        {
            "name": "NGCP 1 AM (4D)",
            "x": ngcp_profile["date_start"],
            "y": ngcp_profile["recovery_pct"],
            "color": NGCP_COLOR,
            "width": 2.5,
            "mode": "lines+markers",
            "marker_size": 4,
            "shape": "hv",
            "unit": "%",
        },
    ],
    y_axis_title=(
        "Output relative to<br>"
        "pre-Haiyan baseline (%)"
    ),
)

fig_roman_normalized.update_yaxes(type='log',row=2, col=1)
fig_roman_normalized.show()


## 4. Reliability-qualified application

The reliability-qualified branch retains direct DNB-BRDF observations with `MQF == 0`, restricts the analysis to GHSL G7 settlements, and applies a daily spatial 95th-percentile clamp. It then uses the same baseline, four-day aggregation, spatial-completeness rule, and pixel-matched recovery calculation as the benchmark.

This branch asks whether a more conservative observation set produces a more interpretable functional-recovery trajectory, even if fewer composites remain available.


In [ ]:
# ============================================================
# 6.1 RELIABILITY-QUALIFIED PIXEL-MATCHED PROFILE
# ============================================================

direct_dnb = dnb

# Direct observations satisfying MQF == 0 and GHSL G7.
rq_unclipped = direct_dnb.where(
    (mqf == 0)
    & g7_mask
)

# Apply the RQ1-style daily spatial 95th-percentile clamp.
rq_quantile_source = rq_unclipped

# Quantile requires each spatial core dimension to occupy one
# Dask chunk. NumPy-backed arrays pass through unchanged.
if hasattr(rq_unclipped.data, "rechunk"):
    spatial_chunk_axes = {
        rq_unclipped.get_axis_num(dimension): -1
        for dimension in SPATIAL_DIMS
    }

    rq_quantile_source = rq_unclipped.copy(
        data=rq_unclipped.data.rechunk(
            spatial_chunk_axes
        )
    )

rq_daily_p95 = (
    rq_quantile_source
    .quantile(
        RQ_CLIP_PERCENTILE / 100.0,
        dim=SPATIAL_DIMS,
        skipna=True,
    )
    .squeeze(drop=True)
    .compute()
)

rq_cube = xr.where(
    rq_unclipped > rq_daily_p95,
    rq_daily_p95,
    rq_unclipped,
)

(
    rq_profile,
    rq_composites,
    rq_ntl0,
    rq_fixed_mask,
    rq_report,
) = build_pixel_matched_profile(
    cube=rq_cube,
    base_mask=g7_mask,
    method="Reliability-qualified GHSL G7",
    input_band=(
        "Direct DNB-BRDF; MQF == 0; "
        "GHSL G7; daily P95 clamp"
    ),
)

print("Reliability-qualified pixel-matched profile")
print(f"Input: {rq_report['input_band']}")
print(
    f"Baseline: {BASELINE_START.date()} to "
    f"{PRE_EVENT_END.date()}"
)
print(
    "Baseline composites: "
    f"{rq_report['baseline_composites']}"
)
print(
    f"Baseline-lit G7 pixels: "
    f"{rq_report['fixed_pixels']:,}"
)
print(
    "Median baseline coverage: "
    f"{rq_report['median_baseline_coverage_pct']:.1f}%"
)
print(
    "Minimum retained spatial completeness: "
    f"{SPATIAL_COMPLETENESS_PCT:.0f}%"
)

display(rq_profile.head())


In [ ]:
# ============================================================
# 6.2 PREPARE RELIABILITY-QUALIFIED DIAGNOSTICS
# ============================================================

rq_daily = extract_daily_ntl_and_sc(
    cube=rq_cube,
    fixed_mask=rq_fixed_mask,
)

rq_profile["raw_mean_ntl"] = (
    rq_profile["current_radiance"]
    / rq_profile["valid_pixel_count"]
)


In [ ]:
# ============================================================
# 8D. RELIABILITY-QUALIFIED PROCESSING SEQUENCE
# ============================================================

# ------------------------------------------------------------
# RQ 1:
# Daily direct, reliability-qualified and gap-filled NTL
# ------------------------------------------------------------

fig_rq_raw = plot_sc_timeseries(
    title=(""
        # "RQ diagnostic 1: daily direct, qualified "
        # "and gap-filled NTL"
    ),
    ntl_panel_title=(""
        # "Daily regional mean radiance on native method support"
    ),
    sc_series=[
        {
            "label": "RQ NTL SC",
            "x": rq_daily.index,
            "y": rq_daily["sc_pct"],
        },
    ],
    line_series=[
        {
            "name": "DNB-BRDF",
            "x": roman_daily.index,
            "y": roman_daily["mean_ntl"],
            "color": DNB_COLOR,
            "width": 1.5,
            "mode": "lines",
            "opacity": 0.75,
            "unit": "nW cm⁻² sr⁻¹",
        },
        {
            "name": "Reliability-qualified NTL",
            "x": rq_daily.index,
            "y": rq_daily["mean_ntl"],
            "color": RQ_COLOR,
            "width": 2.0,
            "mode": "lines+markers",
            "marker_size": 3,
            "unit": "nW cm⁻² sr⁻¹",
        },
        {
            "name": "Gap-filled",
            "x": gap_filled_daily.index,
            "y": gap_filled_daily["mean_ntl"],
            "color": GAP_FILLED_COLOR,
            "width": 1.8,
            "mode": "lines",
            "opacity": 0.75,
            "unit": "nW cm⁻² sr⁻¹",
        },
    ],
    y_axis_title=(
        "Mean DNB-BRDF<br>"
        "(nW cm⁻² sr⁻¹)"
    ),
)

fig_rq_raw.update_yaxes(type='log',row=2, col=1)
fig_rq_raw.show()


**Interpretation.** Reliability qualification removes unstable observations and makes missingness more visible. The resulting gaps indicate limited observability; they should not be interpreted as continued outage.


In [ ]:
# ------------------------------------------------------------
# RQ 2:
# Raw daily RQ DNB-BRDF versus four-day aggregation
# ------------------------------------------------------------

fig_rq_aggregation = plot_sc_timeseries(
    title="",
    ntl_panel_title=(""
        # "Raw daily RQ DNB-BRDF versus four-day mean"
    ),
    sc_series=[
        {
            "label": "RQ NTL SC",
            "x": rq_profile["date_start"],
            "y": rq_profile["spatial_coverage_pct"],
        },
    ],
    line_series=[
        {
            "name": "Reliability-qualified NTL",
            "x": rq_daily.index,
            "y": rq_daily["mean_ntl"],
            "color": RQ_COLOR,
            "width": 1.4,
            "mode": "lines+markers",
            "marker_size": 3,
            "opacity": 0.65,
            "unit": "nW cm⁻² sr⁻¹",
        },
        {
            "name": "Reliability-qualified NTL (4D)",
            "x": rq_profile["date_start"],
            "y": rq_profile["raw_mean_ntl"],
            "color": RQ_FOUR_DAY_COLOR,
            "width": 2.8,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "nW cm⁻² sr⁻¹",
        },
    ],
    y_axis_title=(
        "Mean DNB-BRDF<br>"
        "(nW cm⁻² sr⁻¹)"
    ),
)

fig_rq_aggregation.show()


In [ ]:
# ------------------------------------------------------------
# RQ 3:
# Normalized DNB, RQ DNB and NGCP with RQ SC
# ------------------------------------------------------------

fig_rq_normalized = plot_sc_timeseries(
    title=(""
        # "RQ diagnostic 3: baseline-normalized comparison"
    ),
    ntl_panel_title=(""
        # "Four-day direct DNB, RQ DNB and NGCP"
    ),
    sc_series=[
        {
            "label": "RQ NTL SC",
            "x": rq_profile["date_start"],
            "y": rq_profile[
                "spatial_coverage_pct"
            ],
        },
    ],
    line_series=[
        {
            "name": "DNB-BRDF (4D)",
            "x": roman_profile["date_start"],
            "y": roman_profile["recovery_pct"],
            "color": DNB_FOUR_DAY_COLOR,
            "width": 2.5,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "%",
        },
        {
            "name": "Reliability-qualified NTL (4D)",
            "x": rq_profile["date_start"],
            "y": rq_profile["recovery_pct"],
            "color": RQ_FOUR_DAY_COLOR,
            "width": 2.8,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "%",
        },
        {
            "name": "NGCP 1 AM (4D)",
            "x": ngcp_profile["date_start"],
            "y": ngcp_profile["recovery_pct"],
            "color": NGCP_COLOR,
            "width": 2.5,
            "mode": "lines+markers",
            "marker_size": 4,
            "shape": "hv",
            "unit": "%",
        },
    ],
    y_axis_title=(
        "Output relative to<br>"
        "pre-Haiyan baseline (%)"
    ),
)

fig_rq_normalized.update_yaxes(type='log',row=2, col=1)
fig_rq_normalized.show()


## 5. Cross-comparison with NGCP demand

The two NTL trajectories are compared with NGCP Hour 1 demand using identical four-day periods and the same pre-Haiyan normalization. Agreement indicates that the satellite signal is consistent with broad electricity-dependent activity. Divergence remains meaningful because upward radiance and electricity demand measure related but non-equivalent processes.


In [ ]:
# ============================================================
# 8E. FINAL COMPARISON WITH BOTH SC SERIES
# ============================================================

fig_final_comparison = plot_sc_timeseries(
    title=(""
        # "Final four-day NTL and NGCP comparison"
    ),
    ntl_panel_title=(""
        # "Baseline-normalized direct DNB, RQ DNB and NGCP"
    ),
    sc_series=[
        {
            "label": "DNB-BRDF SC",
            "x": roman_profile["date_start"],
            "y": roman_profile[
                "spatial_coverage_pct"
            ],
        },
        {
            "label": "RQ NTL SC",
            "x": rq_profile["date_start"],
            "y": rq_profile[
                "spatial_coverage_pct"
            ],
        },
    ],
    line_series=[
        {
            "name": "DNB-BRDF (4D)",
            "x": roman_profile["date_start"],
            "y": roman_profile["recovery_pct"],
            "color": DNB_FOUR_DAY_COLOR,
            "width": 2.5,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "%",
        },
        {
            "name": "Reliability-qualified NTL (4D)",
            "x": rq_profile["date_start"],
            "y": rq_profile["recovery_pct"],
            "color": RQ_FOUR_DAY_COLOR,
            "width": 2.8,
            "mode": "lines+markers",
            "marker_size": 5,
            "shape": "hv",
            "unit": "%",
        },
        {
            "name": "NGCP 1 AM (4D)",
            "x": ngcp_profile["date_start"],
            "y": ngcp_profile["recovery_pct"],
            "color": NGCP_COLOR,
            "width": 2.5,
            "mode": "lines+markers",
            "marker_size": 4,
            "shape": "hv",
            "unit": "%",
        },
    ],
    y_axis_title=(
        "Output relative to<br>"
        "pre-Haiyan baseline (%)"
    ),
)

fig_final_comparison.update_yaxes(type='log',row=2, col=1)
fig_final_comparison.show()


In [ ]:
# ============================================================
# 8.2 CALCULATE PHASE-SPECIFIC RESULTS
# ============================================================

# Half-open intervals prevent overlap between phases.
summary_periods = [
    (
        "Baseline",
        pd.Timestamp(BASELINE_START),
        pd.Timestamp(EVENT_DATE),
    ),
    (
        "Stage 1 (0–59 days)",
        pd.Timestamp(EVENT_DATE),
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=60),
    ),
    (
        "Stage 2 (60–119 days)",
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=60),
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=120),
    ),
    (
        "Stage 3 (120–179 days)",
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=120),
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=180),
    ),
    (
        "Post-event (0–179 days)",
        pd.Timestamp(EVENT_DATE),
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=180),
    ),
    (
        "Full analysis",
        pd.Timestamp(BASELINE_START),
        pd.Timestamp(EVENT_DATE)
        + pd.Timedelta(days=180),
    ),
]


# ------------------------------------------------------------
# Prepare the NGCP comparator
# ------------------------------------------------------------

ngcp_summary = (
    ngcp_profile[
        [
            "date_start",
            "recovery_pct",
        ]
    ]
    .rename(
        columns={
            "recovery_pct": "ngcp_recovery_pct",
        }
    )
    .copy()
)

ngcp_summary["date_start"] = pd.to_datetime(
    ngcp_summary["date_start"]
).dt.normalize()

ngcp_summary = (
    ngcp_summary
    .drop_duplicates(
        subset="date_start",
    )
    .sort_values("date_start")
)


# ------------------------------------------------------------
# Safe descriptive and agreement metrics
# ------------------------------------------------------------

def safe_correlation(
    x,
    y,
    method="pearson",
):
    paired_values = pd.DataFrame(
        {
            "x": pd.to_numeric(
                x,
                errors="coerce",
            ),
            "y": pd.to_numeric(
                y,
                errors="coerce",
            ),
        }
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    ).dropna()

    if (
        len(paired_values) < 2
        or paired_values["x"].nunique() < 2
        or paired_values["y"].nunique() < 2
    ):
        return np.nan

    return paired_values["x"].corr(
        paired_values["y"],
        method=method,
    )


def safe_mean(values):
    values = pd.to_numeric(
        values,
        errors="coerce",
    )

    return (
        float(values.mean())
        if values.notna().any()
        else np.nan
    )


def safe_median(values):
    values = pd.to_numeric(
        values,
        errors="coerce",
    )

    return (
        float(values.median())
        if values.notna().any()
        else np.nan
    )


# ------------------------------------------------------------
# Calculate results by method and phase
# ------------------------------------------------------------

summary_rows = []

for method_name, method_profile in [
    (
        "Román-style direct DNB-BRDF",
        roman_profile,
    ),
    (
        "Reliability-qualified DNB-BRDF",
        rq_profile,
    ),
]:
    method_summary = (
        method_profile[
            [
                "date_start",
                "recovery_pct",
                "spatial_coverage_pct",
            ]
        ]
        .rename(
            columns={
                "recovery_pct": "ntl_recovery_pct",
                "spatial_coverage_pct": "sc_pct",
            }
        )
        .copy()
    )

    method_summary["date_start"] = pd.to_datetime(
        method_summary["date_start"]
    ).dt.normalize()

    method_summary = (
        method_summary
        .drop_duplicates(
            subset="date_start",
        )
        .sort_values("date_start")
    )

    for (
        period_name,
        period_start,
        period_end,
    ) in summary_periods:

        method_period = method_summary.loc[
            (
                method_summary["date_start"]
                >= period_start
            )
            & (
                method_summary["date_start"]
                < period_end
            )
        ].copy()

        ngcp_period = ngcp_summary.loc[
            (
                ngcp_summary["date_start"]
                >= period_start
            )
            & (
                ngcp_summary["date_start"]
                < period_end
            )
        ].copy()

        paired_period = (
            method_period
            .merge(
                ngcp_period,
                on="date_start",
                how="inner",
            )
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna(
                subset=[
                    "ntl_recovery_pct",
                    "ngcp_recovery_pct",
                ]
            )
        )

        ntl_values = pd.to_numeric(
            method_period["ntl_recovery_pct"],
            errors="coerce",
        ).dropna()

        sc_values = pd.to_numeric(
            method_period["sc_pct"],
            errors="coerce",
        ).dropna()

        ngcp_values = pd.to_numeric(
            ngcp_period["ngcp_recovery_pct"],
            errors="coerce",
        ).dropna()

        differences = (
            paired_period["ntl_recovery_pct"]
            - paired_period["ngcp_recovery_pct"]
        )

        expected_composites = int(
            np.ceil(
                (
                    period_end
                    - period_start
                ).days
                / ROMAN_BLOCK_DAYS
            )
        )

        retained_composites = int(
            ntl_values.shape[0]
        )

        retained_pct = (
            100.0
            * retained_composites
            / expected_composites
            if expected_composites > 0
            else np.nan
        )

        summary_rows.append(
            {
                "Method": method_name,
                "Period": period_name,
                "Expected n": expected_composites,
                "NTL n": retained_composites,
                "NGCP n": int(
                    ngcp_values.shape[0]
                ),
                "Paired n": int(
                    paired_period.shape[0]
                ),
                "Retained (%)": retained_pct,
                "Mean NTL recovery (%)": (
                    safe_mean(ntl_values)
                ),
                "Median NTL recovery (%)": (
                    safe_median(ntl_values)
                ),
                "NTL recovery SD (%)": (
                    float(ntl_values.std(ddof=1))
                    if len(ntl_values) > 1
                    else np.nan
                ),
                "Minimum NTL recovery (%)": (
                    float(ntl_values.min())
                    if len(ntl_values) > 0
                    else np.nan
                ),
                "Maximum NTL recovery (%)": (
                    float(ntl_values.max())
                    if len(ntl_values) > 0
                    else np.nan
                ),
                "Mean NGCP recovery (%)": (
                    safe_mean(ngcp_values)
                ),
                "Median NGCP recovery (%)": (
                    safe_median(ngcp_values)
                ),
                "Pearson r": safe_correlation(
                    paired_period[
                        "ntl_recovery_pct"
                    ],
                    paired_period[
                        "ngcp_recovery_pct"
                    ],
                    method="pearson",
                ),
                "Spearman ρ": safe_correlation(
                    paired_period[
                        "ntl_recovery_pct"
                    ],
                    paired_period[
                        "ngcp_recovery_pct"
                    ],
                    method="spearman",
                ),
                "MAE vs NGCP (%)": (
                    float(
                        np.abs(differences).mean()
                    )
                    if len(differences) > 0
                    else np.nan
                ),
                "RMSE vs NGCP (%)": (
                    float(
                        np.sqrt(
                            np.mean(
                                differences ** 2
                            )
                        )
                    )
                    if len(differences) > 0
                    else np.nan
                ),
                "Mean bias vs NGCP (%)": (
                    safe_mean(differences)
                ),
                "Mean SC (%)": (
                    safe_mean(sc_values)
                ),
                "Median SC (%)": (
                    safe_median(sc_values)
                ),
                "Minimum SC (%)": (
                    float(sc_values.min())
                    if len(sc_values) > 0
                    else np.nan
                ),
            }
        )


results_summary_table = pd.DataFrame(
    summary_rows
)

period_order = [
    period[0]
    for period in summary_periods
]

results_summary_table["Period"] = pd.Categorical(
    results_summary_table["Period"],
    categories=period_order,
    ordered=True,
)

results_summary_table = (
    results_summary_table
    .sort_values(
        [
            "Method",
            "Period",
        ]
    )
    .reset_index(drop=True)
)


### 5.1 Observability and phase summaries

The tables separate data retention, recovery magnitude, variability, association, and error. This order preserves the analysis logic: observability first, interpretability second, and recovery comparison third.


In [ ]:
# ------------------------------------------------------------
# Display focused summary tables
# ------------------------------------------------------------

core_periods = [
    "Baseline",
    "Stage 1 (0–59 days)",
    "Stage 2 (60–119 days)",
    "Stage 3 (120–179 days)",
]

comparison_periods = core_periods + [
    "Post-event (0–179 days)",
    "Full analysis",
]


def display_results_table(
    table,
    caption,
    formats,
):
    styled_table = (
        table
        .style
        .format(
            formats,
            na_rep="—",
        )
        .hide(axis="index")
        .set_caption(caption)
        .set_properties(
            **{
                "text-align": "center",
                "white-space": "nowrap",
            }
        )
        .set_table_styles(
            [
                {
                    "selector": "caption",
                    "props": [
                        ("font-size", "16px"),
                        ("font-weight", "bold"),
                        ("text-align", "left"),
                        ("margin-bottom", "8px"),
                    ],
                },
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#243B5A"),
                        ("color", "white"),
                        ("text-align", "center"),
                    ],
                },
            ]
        )
    )

    display(styled_table)


# ============================================================
# TABLE 1. OBSERVABILITY AND DATA RETENTION
# ============================================================

observability_table = (
    results_summary_table.loc[
        results_summary_table["Period"].isin(
            comparison_periods
        ),
        [
            "Method",
            "Period",
            "Expected n",
            "NTL n",
            "Paired n",
            "Retained (%)",
            "Mean SC (%)",
            "Median SC (%)",
            "Minimum SC (%)",
        ],
    ]
    .rename(
        columns={
            "Expected n": "Expected composites",
            "NTL n": "Retained NTL composites",
            "Paired n": "Paired with NGCP",
        }
    )
    .reset_index(drop=True)
)

display_results_table(
    table=observability_table,
    caption=(
        "Table 1. Four-day composite availability and "
        "spatial completeness"
    ),
    formats={
        "Expected composites": "{:.0f}",
        "Retained NTL composites": "{:.0f}",
        "Paired with NGCP": "{:.0f}",
        "Retained (%)": "{:.1f}",
        "Mean SC (%)": "{:.1f}",
        "Median SC (%)": "{:.1f}",
        "Minimum SC (%)": "{:.1f}",
    },
)


In [ ]:
# ============================================================
# TABLE 2. MEDIAN RECOVERY BY PHASE
# ============================================================

phase_ntl_recovery = (
    results_summary_table.loc[
        results_summary_table["Period"].isin(
            core_periods
        ),
        [
            "Method",
            "Period",
            "Median NTL recovery (%)",
        ],
    ]
    .pivot(
        index="Method",
        columns="Period",
        values="Median NTL recovery (%)",
    )
    .reindex(columns=core_periods)
)

phase_ngcp_recovery = (
    results_summary_table.loc[
        results_summary_table["Period"].isin(
            core_periods
        )
    ]
    .groupby(
        "Period",
        observed=True,
    )["Median NGCP recovery (%)"]
    .first()
    .reindex(core_periods)
    .to_frame()
    .T
)

phase_ngcp_recovery.index = [
    "NGCP 1 AM demand"
]

phase_recovery_table = (
    pd.concat(
        [
            phase_ntl_recovery,
            phase_ngcp_recovery,
        ],
        axis=0,
    )
    .rename_axis("Series")
    .reset_index()
)

display_results_table(
    table=phase_recovery_table,
    caption=(
        "Table 2. Median output relative to the "
        "pre-Haiyan baseline by phase (%)"
    ),
    formats={
        period: "{:.1f}"
        for period in core_periods
    },
)


In [ ]:
# ============================================================
# TABLES 3–4. SIDE-BY-SIDE METHOD COMPARISONS
# ============================================================

period_labels = {
    "Baseline": "Baseline",
    "Stage 1 (0–59 days)": "Stage 1 (0–59 d)",
    "Stage 2 (60–119 days)": "Stage 2 (60–119 d)",
    "Stage 3 (120–179 days)": "Stage 3 (120–179 d)",
    "Post-event (0–179 days)": "Post-event (0–179 d)",
    "Full analysis": "Full analysis",
}

method_labels = {
    "Román-style direct DNB-BRDF": "Román-style",
    "Reliability-qualified DNB-BRDF": "Reliability-qualified",
}


def build_wide_comparison_table(
    periods,
    metric_columns,
    metric_labels,
):
    table_source = results_summary_table.loc[
        results_summary_table["Period"].isin(
            periods
        ),
        [
            "Method",
            "Period",
            *metric_columns,
        ],
    ].copy()

    table_source["Method"] = table_source[
        "Method"
    ].replace(method_labels)

    table_source["Period"] = (
        table_source["Period"]
        .astype(str)
        .replace(period_labels)
    )

    wide_table = table_source.pivot(
        index="Method",
        columns="Period",
        values=metric_columns,
    )

    # Pivot creates Metric → Period.
    # Reverse this to Period → Metric.
    wide_table = wide_table.swaplevel(
        0,
        1,
        axis=1,
    )

    wide_table.columns = pd.MultiIndex.from_tuples(
        [
            (
                period,
                metric_labels[metric],
            )
            for period, metric in wide_table.columns
        ],
        names=[
            "Period",
            "Metric",
        ],
    )

    ordered_columns = pd.MultiIndex.from_product(
        [
            [
                period_labels[period]
                for period in periods
            ],
            [
                metric_labels[metric]
                for metric in metric_columns
            ],
        ],
        names=[
            "Period",
            "Metric",
        ],
    )

    wide_table = wide_table.reindex(
        columns=ordered_columns
    )

    wide_table = wide_table.reindex(
        [
            "Román-style",
            "Reliability-qualified",
        ]
    )

    wide_table.index.name = "Method"

    return wide_table


def display_wide_comparison_table(
    table,
    caption,
    metric_formats,
):
    column_formats = {
        column: metric_formats[column[1]]
        for column in table.columns
    }

    def shade_method_row(row):
        if row.name == "Román-style":
            background = "#FFF3E6"
        else:
            background = "#E8F5EF"

        return [
            f"background-color: {background}"
            for _ in row
        ]

    styled_table = (
        table
        .style
        .format(
            column_formats,
            na_rep="—",
        )
        .apply(
            shade_method_row,
            axis=1,
        )
        .set_caption(caption)
        .set_properties(
            **{
                "text-align": "center",
                "white-space": "nowrap",
                "border": "1px solid #D5DCE5",
            }
        )
        .set_table_styles(
            [
                {
                    "selector": "caption",
                    "props": [
                        ("font-size", "16px"),
                        ("font-weight", "bold"),
                        ("text-align", "left"),
                        ("margin-bottom", "8px"),
                    ],
                },
                {
                    "selector": "th.col_heading.level0",
                    "props": [
                        ("background-color", "#243B5A"),
                        ("color", "white"),
                        ("font-weight", "bold"),
                        ("text-align", "center"),
                        ("border", "1px solid white"),
                    ],
                },
                {
                    "selector": "th.col_heading.level1",
                    "props": [
                        ("background-color", "#DCE5F0"),
                        ("color", "#243B5A"),
                        ("font-weight", "bold"),
                        ("text-align", "center"),
                        ("border", "1px solid white"),
                    ],
                },
                {
                    "selector": "th.row_heading",
                    "props": [
                        ("background-color", "#F3F5F8"),
                        ("color", "#243B5A"),
                        ("font-weight", "bold"),
                        ("text-align", "left"),
                        ("padding", "6px 10px"),
                    ],
                },
                {
                    "selector": "th.index_name",
                    "props": [
                        ("background-color", "#243B5A"),
                        ("color", "white"),
                        ("font-weight", "bold"),
                        ("text-align", "left"),
                    ],
                },
            ]
        )
    )

    display(styled_table)


# ============================================================
# TABLE 3A. PHASE RECOVERY LEVEL
# ============================================================

table_3a_metrics = [
    "NTL n",
    "Mean NTL recovery (%)",
    "Median NTL recovery (%)",
]

table_3a_labels = {
    "NTL n": "n",
    "Mean NTL recovery (%)": "Mean (%)",
    "Median NTL recovery (%)": "Median (%)",
}

table_3a = build_wide_comparison_table(
    periods=core_periods,
    metric_columns=table_3a_metrics,
    metric_labels=table_3a_labels,
)

display_wide_comparison_table(
    table=table_3a,
    caption=(
        "Table 3a. NTL-derived recovery level by phase"
    ),
    metric_formats={
        "n": "{:.0f}",
        "Mean (%)": "{:.1f}",
        "Median (%)": "{:.1f}",
    },
)


# ============================================================
# TABLE 3B. WITHIN-PHASE VARIABILITY
# ============================================================

table_3b_metrics = [
    "NTL recovery SD (%)",
    "Minimum NTL recovery (%)",
    "Maximum NTL recovery (%)",
]

table_3b_labels = {
    "NTL recovery SD (%)": "SD (%)",
    "Minimum NTL recovery (%)": "Minimum (%)",
    "Maximum NTL recovery (%)": "Maximum (%)",
}

table_3b = build_wide_comparison_table(
    periods=core_periods,
    metric_columns=table_3b_metrics,
    metric_labels=table_3b_labels,
)

display_wide_comparison_table(
    table=table_3b,
    caption=(
        "Table 3b. Within-phase variability of "
        "NTL-derived recovery"
    ),
    metric_formats={
        "SD (%)": "{:.1f}",
        "Minimum (%)": "{:.1f}",
        "Maximum (%)": "{:.1f}",
    },
)


# ============================================================
# TABLE 4A. PHASE-SPECIFIC ASSOCIATION WITH NGCP
# ============================================================

table_4a_metrics = [
    "Paired n",
    "Pearson r",
    "Spearman ρ",
]

table_4a_labels = {
    "Paired n": "n",
    "Pearson r": "Pearson r",
    "Spearman ρ": "Spearman ρ",
}

table_4a = build_wide_comparison_table(
    periods=core_periods,
    metric_columns=table_4a_metrics,
    metric_labels=table_4a_labels,
)

display_wide_comparison_table(
    table=table_4a,
    caption=(
        "Table 4a. Phase-specific association between "
        "NTL recovery and NGCP demand"
    ),
    metric_formats={
        "n": "{:.0f}",
        "Pearson r": "{:.3f}",
        "Spearman ρ": "{:.3f}",
    },
)


# ============================================================
# TABLE 4B. PHASE-SPECIFIC ERROR RELATIVE TO NGCP
# ============================================================

table_4b_metrics = [
    "MAE vs NGCP (%)",
    "RMSE vs NGCP (%)",
    "Mean bias vs NGCP (%)",
]

table_4b_labels = {
    "MAE vs NGCP (%)": "MAE (%)",
    "RMSE vs NGCP (%)": "RMSE (%)",
    "Mean bias vs NGCP (%)": "Mean bias (%)",
}

table_4b = build_wide_comparison_table(
    periods=core_periods,
    metric_columns=table_4b_metrics,
    metric_labels=table_4b_labels,
)

display_wide_comparison_table(
    table=table_4b,
    caption=(
        "Table 4b. Phase-specific NTL error relative "
        "to NGCP demand"
    ),
    metric_formats={
        "MAE (%)": "{:.1f}",
        "RMSE (%)": "{:.1f}",
        "Mean bias (%)": "{:+.1f}",
    },
)


# ============================================================
# TABLE 4C. OVERALL POST-EVENT AND FULL-PERIOD AGREEMENT
# ============================================================

overall_periods = [
    "Post-event (0–179 days)",
    "Full analysis",
]

table_4c_metrics = [
    "Paired n",
    "Pearson r",
    "Spearman ρ",
    "MAE vs NGCP (%)",
    "RMSE vs NGCP (%)",
    "Mean bias vs NGCP (%)",
]

table_4c_labels = {
    "Paired n": "n",
    "Pearson r": "Pearson r",
    "Spearman ρ": "Spearman ρ",
    "MAE vs NGCP (%)": "MAE (%)",
    "RMSE vs NGCP (%)": "RMSE (%)",
    "Mean bias vs NGCP (%)": "Mean bias (%)",
}

table_4c = build_wide_comparison_table(
    periods=overall_periods,
    metric_columns=table_4c_metrics,
    metric_labels=table_4c_labels,
)

display_wide_comparison_table(
    table=table_4c,
    caption=(
        "Table 4c. Overall agreement between NTL "
        "recovery and NGCP demand"
    ),
    metric_formats={
        "n": "{:.0f}",
        "Pearson r": "{:.3f}",
        "Spearman ρ": "{:.3f}",
        "MAE (%)": "{:.1f}",
        "RMSE (%)": "{:.1f}",
        "Mean bias (%)": "{:+.1f}",
    },
)


### 5.2 Preliminary interpretation

Four-day aggregation produces a nearly complete benchmark record, but its spatial support is uneven. Reliability qualification retains slightly fewer post-event composites and exposes a substantially weaker baseline.

The reliability-qualified trajectory reproduces the expected shock-recovery ordering and broadly follows NGCP demand across the three post-Haiyan phases. Its phase means and medians remain close, and its post-event association with NGCP is markedly stronger than the unqualified benchmark. However, the reliability-qualified series retains a positive bias relative to NGCP. It should therefore be interpreted as a functional-recovery proxy, not as the percentage of electricity load restored.

The benchmark contains large radiance excursions and unstable phase summaries. These differences indicate that transferring temporal aggregation alone is insufficient under Samar-Leyte observation conditions. They do not prove that any single filter caused the improvement because MQF screening, settlement support, and clipping change together.


## 6. Spatial recovery patterns

The spatial analysis asks where the regional time-series behaviour occurred. Baseline radiance is compared with three fixed 60-day post-Haiyan stages over Tacloban. A common colour scale is used so that brightness differences are comparable across panels.


In [ ]:
# ------------------------------------------------------------
# 10.1 Load the regional boundary
# ------------------------------------------------------------

if not REGION_SHAPEFILE.exists():
    raise FileNotFoundError(
        f"Regional boundary shapefile not found:\n"
        f"{REGION_SHAPEFILE}"
    )

region_boundary = (
    gpd.read_file(REGION_SHAPEFILE)
    .to_crs("EPSG:4326")
)


# ------------------------------------------------------------
# 10.2 Tacloban spatial extent
# ------------------------------------------------------------


def subset_tacloban(data_array):
    """Subset a raster regardless of coordinate direction."""

    x_values = data_array["x"].values
    y_values = data_array["y"].values

    x_slice = (
        slice(*TACLOBAN_X_RANGE)
        if x_values[0] < x_values[-1]
        else slice(*TACLOBAN_X_RANGE[::-1])
    )

    y_slice = (
        slice(*TACLOBAN_Y_RANGE)
        if y_values[0] < y_values[-1]
        else slice(*TACLOBAN_Y_RANGE[::-1])
    )

    return data_array.sel(
        x=x_slice,
        y=y_slice,
    )


# ------------------------------------------------------------
# Build map composites after subsetting to Tacloban. This avoids
# recomputing the full Samar-Leyte cube for every map.
# ------------------------------------------------------------

def build_tacloban_composites(cube, fixed_mask):
    selected = (
        subset_tacloban(
            cube.sel(
                date=slice(BASELINE_START, PROFILE_END)
            )
        )
        .where(subset_tacloban(fixed_mask))
    )

    dates = pd.DatetimeIndex(
        selected["date"].values
    ).normalize()

    block_numbers = np.floor_divide(
        (dates - EVENT_DATE).days,
        ROMAN_BLOCK_DAYS,
    ).astype(int)

    return (
        selected
        .assign_coords(
            block=("date", block_numbers)
        )
        .groupby("block")
        .mean(dim="date", skipna=True)
        .compute()
    )


roman_tacloban_composites = build_tacloban_composites(
    cube=roman_cube,
    fixed_mask=roman_fixed_mask,
)

rq_tacloban_composites = build_tacloban_composites(
    cube=rq_cube,
    fixed_mask=rq_fixed_mask,
)


# ------------------------------------------------------------
# 10.3 Calculate stage-level median NTL
# ------------------------------------------------------------

def calculate_stage_map(
    composites,
    profile,
    stage_start,
    stage_end,
):
    """Median of admissible four-day composites within a stage."""

    eligible = profile.loc[
        (profile["date_start"] >= stage_start)
        & (profile["date_end"] <= stage_end)
        & profile["recovery_pct"].notna()
    ]

    stage_blocks = (
        eligible["block"]
        .astype(int)
        .to_numpy()
    )

    if len(stage_blocks) == 0:
        return xr.full_like(
            composites.isel(block=0),
            np.nan,
        ).compute()

    return (
        subset_tacloban(
            composites.sel(block=stage_blocks)
        )
        .median(
            dim="block",
            skipna=True,
        )
        .compute()
    )


roman_maps = {
    "Baseline": subset_tacloban(roman_ntl0).compute(),
}

rq_maps = {
    "Baseline": subset_tacloban(rq_ntl0).compute(),
}

for stage_name, (
    stage_start,
    stage_end,
) in list(STAGE_WINDOWS.items())[1:]:

    roman_maps[stage_name] = calculate_stage_map(
        composites=roman_tacloban_composites,
        profile=roman_profile,
        stage_start=stage_start,
        stage_end=stage_end,
    )

    rq_maps[stage_name] = calculate_stage_map(
        composites=rq_tacloban_composites,
        profile=rq_profile,
        stage_start=stage_start,
        stage_end=stage_end,
    )


roman_tacloban_maps = {
    stage_name: subset_tacloban(stage_map)
    for stage_name, stage_map in roman_maps.items()
}

rq_tacloban_maps = {
    stage_name: subset_tacloban(stage_map)
    for stage_name, stage_map in rq_maps.items()
}


# ------------------------------------------------------------
# 10.4 Construct the land mask from the shapefile
# ------------------------------------------------------------

reference_map = roman_tacloban_maps["Baseline"]

x_coordinates = reference_map["x"].values
y_coordinates = reference_map["y"].values

longitude_grid, latitude_grid = np.meshgrid(
    x_coordinates,
    y_coordinates,
)

pixel_centres = gpd.GeoSeries(
    gpd.points_from_xy(
        longitude_grid.ravel(),
        latitude_grid.ravel(),
    ),
    crs="EPSG:4326",
)

land_geometry = (
    region_boundary.cx[
        TACLOBAN_X_RANGE[0]:TACLOBAN_X_RANGE[1],
        TACLOBAN_Y_RANGE[0]:TACLOBAN_Y_RANGE[1],
    ]
    .geometry
    .union_all()
)

land_values = (
    pixel_centres
    .intersects(land_geometry)
    .to_numpy()
    .reshape(
        len(y_coordinates),
        len(x_coordinates),
    )
)

land_mask_tacloban = xr.DataArray(
    land_values,
    dims=("y", "x"),
    coords={
        "y": y_coordinates,
        "x": x_coordinates,
    },
)

g7_tacloban = (
    subset_tacloban(g7_mask)
    .fillna(False)
    .astype(bool)
)

# Román: retain every land pixel; no GHSL filtering.
roman_display_maps = {
    stage_name: stage_map.where(
        land_mask_tacloban
    )
    for stage_name, stage_map
    in roman_tacloban_maps.items()
}

# RQ only: retain GHSL G7 land pixels.
rq_display_mask = (
    land_mask_tacloban
    & g7_tacloban
)

rq_display_maps = {
    stage_name: stage_map.where(
        rq_display_mask
    )
    for stage_name, stage_map
    in rq_tacloban_maps.items()
}


# ------------------------------------------------------------
# 10.5 Extract the regional boundary lines
# ------------------------------------------------------------

tacloban_boundary = region_boundary.cx[
    TACLOBAN_X_RANGE[0]:TACLOBAN_X_RANGE[1],
    TACLOBAN_Y_RANGE[0]:TACLOBAN_Y_RANGE[1],
].copy()


def extract_boundary_lines(geodataframe):
    """Return polygon exterior coordinates for Plotly."""

    boundary_lines = []

    for geometry in geodataframe.geometry:
        if geometry is None or geometry.is_empty:
            continue

        if geometry.geom_type == "Polygon":
            polygons = [geometry]

        elif geometry.geom_type == "MultiPolygon":
            polygons = list(geometry.geoms)

        else:
            continue

        for polygon in polygons:
            longitude, latitude = (
                polygon.exterior.xy
            )

            boundary_lines.append(
                (
                    np.asarray(longitude),
                    np.asarray(latitude),
                )
            )

    return boundary_lines


boundary_lines = extract_boundary_lines(
    tacloban_boundary
)


In [ ]:
# ------------------------------------------------------------
# 10.6 Universal shared radiance scale
# ------------------------------------------------------------

all_map_values = []

for map_collection in (
    roman_display_maps,
    rq_display_maps,
):
    for stage_map in map_collection.values():
        values = np.asarray(
            stage_map.values,
            dtype=float,
        )

        values = values[
            np.isfinite(values)
        ]

        if values.size > 0:
            all_map_values.append(values)

if not all_map_values:
    raise ValueError(
        "No valid Tacloban NTL values were available."
    )

all_map_values = np.concatenate(
    all_map_values
)

radiance_color_max = max(
    float(
        np.nanpercentile(
            all_map_values,
            98,
        )
    ),
    1.0,
)

print(
    "Universal radiance range: "
    f"0–{radiance_color_max:.2f} nW cm⁻² sr⁻¹"
)


# ------------------------------------------------------------
# 10.7 Plot baseline and stage maps
# ------------------------------------------------------------

stage_names = list(STAGE_WINDOWS.keys())

subplot_titles = [
    f"Román | {stage_name}"
    for stage_name in stage_names
] + [
    f"RQ | {stage_name}"
    for stage_name in stage_names
]

fig_technical_maps = make_subplots(
    rows=2,
    cols=4,
    horizontal_spacing=0.035,
    vertical_spacing=0.10,
    subplot_titles=subplot_titles,
)

# Used only under the RQ row.
# Land is gray; water remains transparent/white.
land_background = np.where(
    land_mask_tacloban.values,
    1.0,
    np.nan,
)

for row_number, map_collection in (
    (1, roman_display_maps),
    (2, rq_display_maps),
):
    for column_number, stage_name in enumerate(
        stage_names,
        start=1,
    ):
        stage_map = map_collection[stage_name]

        # Gray non-G7 land background for RQ only.
        if row_number == 2:
            fig_technical_maps.add_trace(
                go.Heatmap(
                    x=x_coordinates,
                    y=y_coordinates,
                    z=land_background,
                    colorscale=[
                        [0.0, "#C8C8C8"],
                        [1.0, "#C8C8C8"],
                    ],
                    zmin=0,
                    zmax=1,
                    showscale=False,
                    hoverinfo="skip",
                ),
                row=row_number,
                col=column_number,
            )

        # NTL radiance
        hover_text = np.where(
            np.isfinite(stage_map.values),
            np.round(stage_map.values, 2).astype(str),
            "",
        )

        fig_technical_maps.add_trace(
            go.Heatmap(
                x=stage_map["x"].values,
                y=stage_map["y"].values,
                z=stage_map.values,
                text=hover_text,
                coloraxis="coloraxis",
                zsmooth=False,
                hovertemplate=(
                    "Longitude: %{x:.4f}<br>"
                    "Latitude: %{y:.4f}<br>"
                    "NTL: %{text} nW cm⁻² sr⁻¹"
                    "<extra></extra>"
                ),
            ),
            row=row_number,
            col=column_number,
        )

        # Shapefile boundary overlay
        for longitude, latitude in boundary_lines:
            fig_technical_maps.add_trace(
                go.Scatter(
                    x=longitude,
                    y=latitude,
                    mode="lines",
                    line=dict(
                        color="#303030",
                        width=1.3,
                    ),
                    hoverinfo="skip",
                    showlegend=False,
                ),
                row=row_number,
                col=column_number,
            )

        axis_number = (
            (row_number - 1) * 4
            + column_number
        )

        x_axis_reference = (
            "x"
            if axis_number == 1
            else f"x{axis_number}"
        )

        fig_technical_maps.update_xaxes(
            range=list(TACLOBAN_X_RANGE),
            showgrid=False,
            zeroline=False,
            row=row_number,
            col=column_number,
        )

        fig_technical_maps.update_yaxes(
            range=list(TACLOBAN_Y_RANGE),
            scaleanchor=x_axis_reference,
            scaleratio=1,
            showgrid=False,
            zeroline=False,
            row=row_number,
            col=column_number,
        )

fig_technical_maps.update_xaxes(
    title_text="Longitude",
    row=2,
)

fig_technical_maps.update_yaxes(
    title_text="Latitude",
    col=1,
)

fig_technical_maps.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1200,
    height=600,
    coloraxis=dict(
        colorscale="Inferno",
        cmin=0,
        cmax=radiance_color_max,
        colorbar=dict(
            title=dict(
                text="NTL<br>nW cm⁻² sr⁻¹"
            ),
            x=1.015,
            xanchor="left",
            y=0.5,
            len=0.82,
            thickness=20,
            outlinewidth=0.8,
            outlinecolor="#555555",
        ),
    ),
    font=dict(
        family="Arial",
        size=13,
        color="#243B5A",
    ),
    margin=dict(
        l=70,
        r=130,
        t=135,
        b=70,
    ),
)

fig_technical_maps.show()


### 6.1 Four-day spatial progression

The phase medians summarize recovery, while the following progression retains all fifteen four-day composites within each phase. These diagnostic grids show whether a stage-level pattern is persistent or driven by a small number of observations.


In [ ]:
# ============================================================
# 10.8 FOUR-DAY COMPOSITE PROGRESSION BY PHASE
# ============================================================

# Each row contains 15 consecutive four-day composites.
progression_rows = [
    (
        "Pre-event baseline\n(−60 to −1 d)",
        list(range(-15, 0)),
    ),
    (
        "Stage 1\n(0–59 d)",
        list(range(0, 15)),
    ),
    (
        "Stage 2\n(60–119 d)",
        list(range(15, 30)),
    ),
    (
        "Stage 3\n(120–179 d)",
        list(range(30, 45)),
    ),
]


# ------------------------------------------------------------
# Display masks
# ------------------------------------------------------------

roman_progression_mask = (
    land_mask_tacloban
    & subset_tacloban(
        roman_fixed_mask
    )
    .fillna(False)
    .astype(bool)
)

rq_progression_mask = (
    land_mask_tacloban
    & g7_tacloban
    & subset_tacloban(
        rq_fixed_mask
    )
    .fillna(False)
    .astype(bool)
)

progression_land_background = np.where(
    land_mask_tacloban.values,
    1.0,
    np.nan,
)


# ------------------------------------------------------------
# Prepare one composite map
# ------------------------------------------------------------

def prepare_progression_map(
    composite_cube,
    block_number,
    display_mask,
):
    """Extract and mask one Tacloban four-day composite."""

    available_blocks = np.asarray(
        composite_cube["block"].values,
        dtype=int,
    )

    if block_number not in available_blocks:
        return None

    composite_map = (
        subset_tacloban(
            composite_cube.sel(
                block=block_number
            )
        )
        .where(display_mask)
    )

    if hasattr(
        composite_map.data,
        "compute",
    ):
        composite_map = (
            composite_map.compute()
        )

    return composite_map


# ------------------------------------------------------------
# Prepare all Román and RQ snapshots
# ------------------------------------------------------------

roman_progression_maps = []
rq_progression_maps = []

for _, block_numbers in progression_rows:
    roman_progression_maps.append(
        [
            prepare_progression_map(
                composite_cube=roman_tacloban_composites,
                block_number=block_number,
                display_mask=roman_progression_mask,
            )
            for block_number in block_numbers
        ]
    )

    rq_progression_maps.append(
        [
            prepare_progression_map(
                composite_cube=rq_tacloban_composites,
                block_number=block_number,
                display_mask=rq_progression_mask,
            )
            for block_number in block_numbers
        ]
    )


# ------------------------------------------------------------
# One radiance scale shared by both figures
# ------------------------------------------------------------

progression_values = []

for map_grid in (
    roman_progression_maps,
    rq_progression_maps,
):
    for map_row in map_grid:
        for composite_map in map_row:
            if composite_map is None:
                continue

            values = np.asarray(
                composite_map.values,
                dtype=float,
            )

            values = values[
                np.isfinite(values)
            ]

            if values.size > 0:
                progression_values.append(
                    values
                )

if not progression_values:
    raise ValueError(
        "No valid four-day Tacloban maps "
        "were available."
    )

progression_values = np.concatenate(
    progression_values
)

progression_color_max = max(
    float(
        np.nanpercentile(
            progression_values,
            98,
        )
    ),
    1.0,
)

print(
    "Common progression radiance range: "
    f"0–{progression_color_max:.2f} "
    "nW cm⁻² sr⁻¹"
)


# ------------------------------------------------------------
# Combine boundary segments into one trace per panel
# ------------------------------------------------------------

progression_boundary_x = []
progression_boundary_y = []

for longitude, latitude in boundary_lines:
    progression_boundary_x.extend(
        [
            *longitude.tolist(),
            None,
        ]
    )

    progression_boundary_y.extend(
        [
            *latitude.tolist(),
            None,
        ]
    )


# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------

def plot_composite_progression(
    map_grid,
    figure_title,
):
    """Plot 15 four-day snapshots for each analysis phase."""

    figure = make_subplots(
        rows=4,
        cols=15,
        shared_xaxes=True,
        shared_yaxes=True,
        horizontal_spacing=0.002,
        vertical_spacing=0.025,
        column_titles=[
            str(column_number)
            for column_number in range(
                1,
                16,
            )
        ],
    )

    longitude_centre = np.mean(
        TACLOBAN_X_RANGE
    )

    latitude_centre = np.mean(
        TACLOBAN_Y_RANGE
    )

    for row_number, (
        row_label,
        block_numbers,
    ) in enumerate(
        progression_rows,
        start=1,
    ):
        figure.update_yaxes(
            title_text=row_label,
            title_font=dict(
                size=12,
            ),
            title_standoff=4,
            row=row_number,
            col=1,
        )

        for column_number, block_number in enumerate(
            block_numbers,
            start=1,
        ):
            composite_map = map_grid[
                row_number - 1
            ][
                column_number - 1
            ]

            # Gray land background; water remains white.
            figure.add_trace(
                go.Heatmap(
                    x=x_coordinates,
                    y=y_coordinates,
                    z=progression_land_background,
                    colorscale=[
                        [0.0, "#CCCCCC"],
                        [1.0, "#CCCCCC"],
                    ],
                    zmin=0,
                    zmax=1,
                    showscale=False,
                    hoverinfo="skip",
                ),
                row=row_number,
                col=column_number,
            )

            block_start = (
                EVENT_DATE
                + pd.Timedelta(
                    days=(
                        block_number
                        * ROMAN_BLOCK_DAYS
                    )
                )
            )

            block_end = (
                block_start
                + pd.Timedelta(
                    days=(
                        ROMAN_BLOCK_DAYS
                        - 1
                    )
                )
            )

            if (
                composite_map is not None
                and np.isfinite(
                    composite_map.values
                ).any()
            ):
                figure.add_trace(
                    go.Heatmap(
                        x=composite_map["x"].values,
                        y=composite_map["y"].values,
                        z=composite_map.values,
                        coloraxis="coloraxis",
                        zsmooth=False,
                        hoverongaps=False,
                        hovertemplate=(
                            f"{row_label.replace('<br>', ' ')}"
                            "<br>"
                            f"Composite {column_number}"
                            "<br>"
                            f"{block_start:%d %b %Y}"
                            "–"
                            f"{block_end:%d %b %Y}"
                            "<br>"
                            "DNB-BRDF: "
                            "%{z:.2f} nW cm⁻² sr⁻¹"
                            "<extra></extra>"
                        ),
                    ),
                    row=row_number,
                    col=column_number,
                )

            else:
                figure.add_trace(
                    go.Scatter(
                        x=[longitude_centre],
                        y=[latitude_centre],
                        mode="text",
                        text=["No data"],
                        textfont=dict(
                            family="Arial",
                            size=9,
                            color="#666666",
                        ),
                        hoverinfo="skip",
                        showlegend=False,
                    ),
                    row=row_number,
                    col=column_number,
                )

            # Boundary overlay
            figure.add_trace(
                go.Scatter(
                    x=progression_boundary_x,
                    y=progression_boundary_y,
                    mode="lines",
                    line=dict(
                        color="#303030",
                        width=0.7,
                    ),
                    hoverinfo="skip",
                    showlegend=False,
                ),
                row=row_number,
                col=column_number,
            )

            figure.update_xaxes(
                range=list(
                    TACLOBAN_X_RANGE
                ),
                showticklabels=False,
                ticks="",
                showgrid=False,
                zeroline=False,
                fixedrange=True,
                row=row_number,
                col=column_number,
            )

            figure.update_yaxes(
                range=list(
                    TACLOBAN_Y_RANGE
                ),
                showticklabels=False,
                ticks="",
                showgrid=False,
                zeroline=False,
                fixedrange=True,
                row=row_number,
                col=column_number,
            )

    figure.add_annotation(
        x=0.5,
        y=-0.055,
        xref="paper",
        yref="paper",
        text=(
            "Sequential four-day composite "
            "within each phase"
        ),
        showarrow=False,
        font=dict(
            family="Arial",
            size=14,
            color="#243B5A",
        ),
    )

    figure.update_annotations(
        font=dict(
            family="Arial",
            size=11,
            color="#243B5A",
        ),
    )

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=2600,
        height=900,
        title=dict(
            text=figure_title,
            x=0.5,
            y=0.985,
            xanchor="center",
            font=dict(
                family="Arial",
                size=25,
                color="#243B5A",
            ),
        ),
        coloraxis=dict(
            colorscale="Inferno",
            cmin=0,
            cmax=progression_color_max,
            colorbar=dict(
                title=dict(
                    text=(
                        "DNB-BRDF"
                        "<br>"
                        "nW cm⁻² sr⁻¹"
                    )
                ),
                x=1.005,
                xanchor="left",
                y=0.5,
                len=0.86,
                thickness=20,
                outlinewidth=0.8,
                outlinecolor="#555555",
            ),
        ),
        font=dict(
            family="Arial",
            size=11,
            color="#243B5A",
        ),
        showlegend=False,
        margin=dict(
            l=120,
            r=145,
            t=105,
            b=70,
        ),
        hovermode="closest",
    )

    return figure


# ------------------------------------------------------------
# Román-style progression
# ------------------------------------------------------------

fig_roman_progression = (
    plot_composite_progression(
        map_grid=roman_progression_maps,
        figure_title=(
            "Román-style DNB-BRDF: "
            "four-day progression by phase"
        ),
    )
)

fig_roman_progression.show()


# ------------------------------------------------------------
# Reliability-qualified progression
# ------------------------------------------------------------

fig_rq_progression = (
    plot_composite_progression(
        map_grid=rq_progression_maps,
        figure_title=(
            "Reliability-qualified DNB-BRDF: "
            "four-day progression by phase"
        ),
    )
)

fig_rq_progression.show()


# ------------------------------------------------------------
# Optional static PNG export
# Both images use the same radiance scale.
# ------------------------------------------------------------

# fig_roman_progression.write_image(
#     "roman_15x4_progression.png",
#     scale=2,
# )

# fig_rq_progression.write_image(
#     "rq_15x4_progression.png",
#     scale=2,
# )


## 7. Public-facing figures

The figures in this section reuse the analysis outputs but remove coordinate labels, radiance units, spatial-completeness strips, method acronyms, and technical thresholds. They are designed for a media release or general-audience presentation; the analytical figures above retain the methodological detail.


In [ ]:
# ============================================================
# 10.6 PUBLIC-FACING RQ MAPS
#     Four reliability-qualified maps only
# ============================================================

stage_names = list(STAGE_WINDOWS.keys())

public_labels = [
    "Before Haiyan",
    "0–2 months",
    "2–4 months",
    "4–6 months",
]

if len(stage_names) != len(public_labels):
    raise ValueError(
        "Expected four stages: baseline and three recovery periods."
    )


# ------------------------------------------------------------
# Shared colour range using RQ maps only
# ------------------------------------------------------------

rq_map_values = []

for stage_map in rq_display_maps.values():
    values = np.asarray(
        stage_map.values,
        dtype=float,
    )

    values = values[np.isfinite(values)]

    if values.size > 0:
        rq_map_values.append(values)

if not rq_map_values:
    raise ValueError(
        "No valid Tacloban nighttime-lights values were available."
    )

rq_map_values = np.concatenate(rq_map_values)

radiance_color_max = max(
    float(np.nanpercentile(rq_map_values, 98)),
    1.0,
)


# ------------------------------------------------------------
# Four-panel public-facing figure
# ------------------------------------------------------------

fig_public_maps = make_subplots(
    rows=1,
    cols=4,
    horizontal_spacing=0.012,
    subplot_titles=[
        f"<b>{label}</b>"
        for label in public_labels
    ],
)

# Light-grey land background outside the selected urban pixels.
land_background = np.where(
    land_mask_tacloban.values,
    1.0,
    np.nan,
)

for column_number, stage_name in enumerate(
    stage_names,
    start=1,
):
    stage_map = rq_display_maps[stage_name]

    # Land background
    fig_public_maps.add_trace(
        go.Heatmap(
            x=x_coordinates,
            y=y_coordinates,
            z=land_background,
            colorscale=[
                [0.0, "#E6E6E6"],
                [1.0, "#E6E6E6"],
            ],
            zmin=0,
            zmax=1,
            showscale=False,
            hoverinfo="skip",
        ),
        row=1,
        col=column_number,
    )

    # Nighttime lights
    fig_public_maps.add_trace(
        go.Heatmap(
            x=stage_map["x"].values,
            y=stage_map["y"].values,
            z=stage_map.values,
            coloraxis="coloraxis",
            zsmooth=False,
            hoverinfo="skip",
        ),
        row=1,
        col=column_number,
    )

    # Regional boundary
    for longitude, latitude in boundary_lines:
        fig_public_maps.add_trace(
            go.Scatter(
                x=longitude,
                y=latitude,
                mode="lines",
                line=dict(
                    color="#303030",
                    width=1.2,
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column_number,
        )

    x_axis_reference = (
        "x"
        if column_number == 1
        else f"x{column_number}"
    )

    fig_public_maps.update_xaxes(
        range=list(TACLOBAN_X_RANGE),
        showticklabels=False,
        ticks="",
        title_text=None,
        showgrid=False,
        zeroline=False,
        showline=False,
        fixedrange=True,
        row=1,
        col=column_number,
    )

    fig_public_maps.update_yaxes(
        range=list(TACLOBAN_Y_RANGE),
        showticklabels=False,
        ticks="",
        title_text=None,
        showgrid=False,
        zeroline=False,
        showline=False,
        fixedrange=True,
        scaleanchor=x_axis_reference,
        scaleratio=1,
        row=1,
        col=column_number,
    )


fig_public_maps.update_annotations(
    font=dict(
        family="Arial",
        size=20,
        color="#243B5A",
    )
)

fig_public_maps.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1500,
    height=390,
    coloraxis=dict(
        colorscale="Inferno",
        cmin=0,
        cmax=radiance_color_max,
        colorbar=dict(
            title=dict(
                text="Radiance at night",
                side="right",
                font=dict(size=25),
            ),
            tickmode="array",
            tickvals=[
                0,
                radiance_color_max,
            ],
            ticktext=[
                "Low",
                "High",
            ],
            tickfont=dict(size=15),
            x=1.01,
            xanchor="left",
            y=0.48,
            len=0.9,
            thickness=30,
            outlinewidth=0.8,
            outlinecolor="#555555",
        ),
    ),
    font=dict(
        family="Arial",
        size=13,
        color="#243B5A",
    ),
    margin=dict(
        l=5,
        r=95,
        t=55,
        b=5,
    ),
)

fig_public_maps.show()


In [ ]:
# ============================================================
# 10.2 PUBLIC-FACING RECOVERY TIME SERIES
# ============================================================

fig_public_recovery = go.Figure()

fig_public_recovery.add_trace(
    go.Scatter(
        x=rq_profile["date_start"],
        y=rq_profile["recovery_pct"],
        mode="lines+markers",
        name="Nighttime lights in urban areas",
        connectgaps=False,
        line=dict(
            color="#FF2424",
            width=4,
            shape="hv",
        ),
        marker=dict(size=7),
        hovertemplate=(
            "%{x|%d %b %Y}<br>"
            "Nighttime lights: %{y:.0f}"
            "<extra></extra>"
        ),
    )
)

fig_public_recovery.add_trace(
    go.Scatter(
        x=ngcp_profile["date_start"],
        y=ngcp_profile["recovery_pct"],
        mode="lines+markers",
        name="Recorded electricity demand",
        connectgaps=False,
        line=dict(
            color="#111111",
            width=4,
            shape="hv",
        ),
        marker=dict(size=6),
        hovertemplate=(
            "%{x|%d %b %Y}<br>"
            "Electricity demand: %{y:.0f}"
            "<extra></extra>"
        ),
    )
)

fig_public_recovery.add_hline(
    y=100,
    line=dict(
        color="#7F8C8D",
        width=1.5,
        dash="dot",
    ),
    annotation_text="Before Haiyan",
    annotation_position="top left",
)

fig_public_recovery.add_vline(
    x=EVENT_DATE.to_pydatetime(),
    line=dict(
        color="#2563EB",
        width=2.5,
        dash="dash",
    ),
)

fig_public_recovery.add_annotation(
    x=EVENT_DATE + pd.Timedelta(days=3),
    y=0.96,
    xref="x",
    yref="paper",
    text="Haiyan",
    showarrow=False,
    xanchor="left",
    font=dict(
        family="Arial",
        size=18,
        color="#2563EB",
    ),
)

fig_public_recovery.update_xaxes(
    title_text="Date",
    range=[BASELINE_START, PROFILE_END],
    showgrid=True,
    gridcolor="#E8EDF3",
)

fig_public_recovery.update_yaxes(
    title_text="Relative level (before Haiyan = 100)",
    rangemode="tozero",
    showgrid=True,
    gridcolor="#E8EDF3",
)

fig_public_recovery.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1500,
    height=650,
    title=dict(
        text=(
            "Nighttime lights and electricity demand "
            "after Haiyan"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=25),
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=30),
    ),
    font=dict(
        family="Arial",
        size=17,
        color="#243B5A",
    ),
    margin=dict(
        l=105,
        r=40,
        t=130,
        b=80,
    ),
    hovermode="x unified",
)

fig_public_recovery.show()


## 8. Conclusions and limitations

- The Román-style framework is transferable for detecting Haiyan and describing the broad direction of functional recovery.
- Reliability qualification produces a more stable trajectory and stronger agreement with NGCP demand, while making observation-limited periods explicit.
- NTL and electricity demand remain non-equivalent. Differences may reflect generators, public lighting, activity patterns, relocation, rebuilding, spatial support, or the light-emission environment.
- The current comparison changes several processing choices together. It supports the reliability-qualified workflow as an application framework but does not attribute improvement to one filter.
- A location must be sufficiently observable before its NTL trajectory is interpreted. “Not recovered” and “not observable” remain distinct outcomes.

The next analytical step is to test whether impact and recovery metrics remain identifiable across communities and settlement contexts, then use finer electricity, physical-recovery, hazard, and contextual evidence to explain alignment and divergence.
